<a href="https://colab.research.google.com/github/dennisgathu8/36CHAMBERS/blob/main/RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install wikipedia
!pip install transformers
!pip install faiss-cpu
!pip install numpy

import wikipedia
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [7]:
#retrieving knowledge
def get_wikipedia_content(topic):
  try:
    page = wikipedia.page(topic)
    return page.content
  except wikipedia.exceptions.PageError:
    return None
  except wikipedia.exceptions.DisambiguationError as e:
    # handle cases where the topic is ambigous
    print(f"Ambigous topic. Please be more specific. Options: {e.options}")
    return None

# user input
topic = input("Enter a topic to learn about: ")
document = get_wikipedia_content(topic)

if not document:
  print("Could not retrieve information.")
  exit()

Enter a topic to learn about: Wakadinali


In [8]:
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-mpnet-base-v2")
def split_text(text, chunk_size=256, chunk_overlap=20):
  tokens = tokenizer.tokenize(text)
  chunks = []
  start = 0
  while start < len(tokens):
    end = min(start + chunk_size, len(tokens))
    chunks.append(tokenizer.convert_tokens_to_string(tokens[start:end]))
    if end == len(tokens):
      break
    start = end - chunk_overlap
  return chunks

chunks = split_text(document)
print(f"Number of chunks: {len(chunks)}")

Token indices sequence length is longer than the specified maximum sequence length for this model (660 > 512). Running this sequence through the model will result in indexing errors


Number of chunks: 3


In [10]:
#storing and retrieving knowledge
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
embeddings = embedding_model.encode(chunks)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
#querying the RAG Pipeline
query = input("Ask a question about the topic: ")
query_embedding = embedding_model.encode([query])
k = 3
distances, indices = index.search(np.array(query_embedding), k)
retrieved_chunks = [chunks[i] for i in indices[0]]
print("Retrieved chunks:")
for chunk in retrieved_chunks:
  print("- " + chunk)

Ask a question about the topic: How many albums have been released by Wakadinali
Retrieved chunks:
- 2022, wakadinali were the second most streamed local artists in kenya on spotify. in 2023, wakadinali were the most streamed artists in kenya on spotify. they were also was the fifth most streamed hip - hop artists in kenya. in 2022, wakadinali released three albums. easy and haitaki hasira were released in january 2022, while ndani ya cockpit 3 was released in december of the same year. ndani ya cockpit 3 was the 10th most streamed album of 2023 in kenya. = = discography = = = = = studio albums = = = ndani ya cockpit 1 ( 2017 ) ndani ya cockpit 2 ( 2018 ) victims of madness ( 2020 ) easy ( 2022 ) haitaki hasira ( 2022 ) ndani ya cockpit 3 ( 2022 ) = = awards and recognition = = = = references = =
- sounds and included hit songs like xl, extra pressure and morio anzenza. tangaza magazine writes that the name of the album " invites the audience to look at crime and criminals in a differe

In [13]:
qa_model_name = "deepset/roberta-base-squad2"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)
qa_pipeline = pipeline("question-answering", model=qa_model, tokenizer=qa_tokenizer)

context = " ".join(retrieved_chunks)
answer = qa_pipeline(question=query, context=context)
print(f"Answer: {answer['answer']}")

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Device set to use cpu


Answer: three
